# GOV-01 Version 2 Model Gate: unweighted baseline

This notebook runs the first Version 2 model comparison on the Rome Road Damage Dataset. It uses only training and validation images. The protected test split stays unloaded until a V2 candidate is selected and locked.

The first real model is deliberately **unweighted**. A later experiment will test class weights as one controlled change.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score, precision_score, recall_score, roc_auc_score

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_NAMES = ['No_pothole', 'Pothole']
DATA_DIR = Path('data/processed/road_damage_rome_clean_split')
REPORTS_DIR = Path('reports')
REPORTS_DIR.mkdir(exist_ok=True)

tf.keras.utils.set_random_seed(SEED)

for split_name in ['train', 'validation', 'test']:
    if not (DATA_DIR / split_name).is_dir():
        raise FileNotFoundError(f'Missing clean-split folder: {DATA_DIR / split_name}')

print('Clean V2 split found:', DATA_DIR.resolve())
print('Classes:', CLASS_NAMES)
print('Protected test folder exists but will not be loaded in this notebook.')

In [ ]:
def load_split(split_name, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        DATA_DIR / split_name,
        labels='inferred',
        label_mode='binary',
        class_names=CLASS_NAMES,
        color_mode='rgb',
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=SEED if shuffle else None,
    )

train_raw = load_split('train', shuffle=True)
validation_ds = load_split('validation', shuffle=False)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.03),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
], name='v2_training_augmentation')

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_raw.map(
    lambda images, labels: (data_augmentation(images, training=True), labels),
    num_parallel_calls=AUTOTUNE,
).prefetch(AUTOTUNE)
validation_ds = validation_ds.prefetch(AUTOTUNE)

print('Training augmentation is enabled only for train_ds.')
print('Validation receives no random augmentation.')

In [ ]:
# Naive baseline: always predict the validation majority class, No_pothole.
# It is not trained. A real model should beat it, especially on macro F1.
y_validation = np.concatenate([labels.numpy().ravel() for _, labels in validation_ds])
naive_predictions = np.zeros_like(y_validation, dtype=int)
naive_scores = np.zeros_like(y_validation, dtype=float)

def calculate_metrics(y_true, predictions, scores):
    return {
        'accuracy': float(accuracy_score(y_true, predictions)),
        'macro_f1': float(f1_score(y_true, predictions, average='macro', zero_division=0)),
        'pothole_precision': float(precision_score(y_true, predictions, pos_label=1, zero_division=0)),
        'pothole_recall': float(recall_score(y_true, predictions, pos_label=1, zero_division=0)),
        'no_pothole_recall': float(recall_score(y_true, predictions, pos_label=0, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, scores)),
    }

naive_metrics = calculate_metrics(y_validation, naive_predictions, naive_scores)
display(pd.DataFrame([naive_metrics], index=['v2_naive_no_pothole']))

In [ ]:
# First real model: a compact CNN trained from scratch.
# Class weights are deliberately absent in this first baseline.
cnn_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=IMAGE_SIZE + (3,)),
    tf.keras.layers.Rescaling(1.0 / 255),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(1, activation='sigmoid', name='pothole_probability'),
], name='v2_cnn_unweighted')

cnn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='roc_auc')],
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=4, restore_best_weights=True,
)
cnn_model.summary()

In [ ]:
start_time = time.time()
history = cnn_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=20,
    callbacks=[early_stopping],
    verbose=1,
)
training_seconds = time.time() - start_time
print(f'Training time: {training_seconds:.1f} seconds')

In [ ]:
# Validation evaluation only. Do not replace validation_ds with test data here.
cnn_scores = cnn_model.predict(validation_ds).ravel()
cnn_predictions = (cnn_scores >= 0.50).astype(int)
cnn_metrics = calculate_metrics(y_validation, cnn_predictions, cnn_scores)

comparison = pd.DataFrame([
    {
        'run_name': 'v2_naive_no_pothole',
        'hypothesis': 'Always predicting the validation majority class establishes a reference floor.',
        'changed_factor': 'naive majority rule',
        'class_weight': 'not applicable',
        'training_seconds': 0.0,
        **naive_metrics,
    },
    {
        'run_name': 'v2_cnn_unweighted',
        'hypothesis': 'A compact CNN can learn road-image patterns beyond the naive majority rule.',
        'changed_factor': 'simple CNN model',
        'class_weight': 'none',
        'training_seconds': round(training_seconds, 1),
        **cnn_metrics,
    },
])
comparison.to_csv(REPORTS_DIR / 'v2_experiment_record.csv', index=False)
display(comparison)

print(classification_report(y_validation, cnn_predictions, target_names=CLASS_NAMES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_validation, cnn_predictions, display_labels=CLASS_NAMES)
plt.title('V2 validation confusion matrix: v2_cnn_unweighted')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'v2_cnn_unweighted_validation_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
history_frame = pd.DataFrame(history.history)
history_frame[['loss', 'val_loss']].plot(title='V2 CNN training and validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'v2_cnn_unweighted_learning_curve.png', dpi=150)
plt.show()

print('Saved V2 Colab evidence in reports/:')
for path in sorted(REPORTS_DIR.glob('v2_*')):
    print('-', path.name)

print('Next decision: inspect validation macro F1 and both recalls before adding class weights.')